In [1]:
import os
# os.environ["HUGGINGFACEHUB_API_TOKEN"]

## Install libraries

In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv langchain_huggingface

In [3]:
pip install --upgrade youtube-transcript-api

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFacePipeline
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

##### we can generate transcript of YouTube videos using YT api as there is YT loader too in langchain but that is not that good to generate transcript

## Step 1a - Indexing (Document Ingestion)

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi
import inspect

print("Source file:", inspect.getfile(YouTubeTranscriptApi))
print("Attributes:", dir(YouTubeTranscriptApi))

Source file: /usr/local/lib/python3.11/dist-packages/youtube_transcript_api/_api.py
Attributes: ['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'fetch', 'list']


In [6]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL
from youtube_transcript_api import YouTubeTranscriptApi
# return FetchedTranscript object
# ytt_api = YouTubeTranscriptApi()
# ytt_api.fetch(video_id)

# list of dictionaries
ytt_api = YouTubeTranscriptApi()
fetched_transcript = ytt_api.fetch(video_id)
transcript_list = fetched_transcript.to_raw_data()
# # is iterable
# for snippet in fetched_transcript:
#     print(snippet.text)

# # indexable
# last_snippet = fetched_transcript[-1]

# # provides a length
# snippet_count = len(fetched_transcript)

In [7]:
transcript_list

[{'text': 'the following is a conversation with',
  'start': 0.08,
  'duration': 3.44},
 {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96},
 {'text': 'ceo and co-founder of deepmind', 'start': 3.52, 'duration': 5.119},
 {'text': 'a company that has published and builds',
  'start': 6.72,
  'duration': 4.48},
 {'text': 'some of the most incredible artificial',
  'start': 8.639,
  'duration': 4.561},
 {'text': 'intelligence systems in the history of',
  'start': 11.2,
  'duration': 4.8},
 {'text': 'computing including alfred zero that',
  'start': 13.2,
  'duration': 3.68},
 {'text': 'learned', 'start': 16.0, 'duration': 2.96},
 {'text': 'all by itself to play the game of gold',
  'start': 16.88,
  'duration': 4.559},
 {'text': 'better than any human in the world and',
  'start': 18.96,
  'duration': 5.6},
 {'text': 'alpha fold two that solved protein',
  'start': 21.439,
  'duration': 4.241},
 {'text': 'folding', 'start': 24.56, 'duration': 4.16},
 {'text': 'both tasks consider

In [8]:
transcript = " ".join(chunk["text"] for chunk in transcript_list)

## Step 1b - Indexing (Text Splitting)

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [10]:
len(chunks)

168

In [11]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [12]:
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

embeddings = HuggingFaceEndpointEmbeddings(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    task="feature-extraction"
    # huggingfacehub_api_token=api_key
)

vector_store = FAISS.from_documents(chunks, embeddings)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [13]:
vector_store.index_to_docstore_id

{0: '1f66f785-9d72-4d90-b586-a535929a9151',
 1: 'cd97d314-ba7a-4a7b-b15c-6d9ea95405cd',
 2: 'd92ec0d9-633b-475d-985a-85568a98775b',
 3: 'f2ea1ee4-3869-4477-a09e-18bf0c7f49b6',
 4: 'a829105f-87f9-44b1-a9e4-f429ea617501',
 5: '5f3bb04a-8975-4045-a872-324317964f41',
 6: '2f5ae086-52ac-4b09-9fa9-2afa367b90b6',
 7: '552c975c-9ce3-4288-a36d-28850ff9e5a8',
 8: '07577f88-9cbf-412e-9ae6-3a621202fe71',
 9: '695a992e-ce1d-4628-9309-a2f6ae836493',
 10: '4a94f3e6-572f-46b1-b4a6-0f13d0da7196',
 11: '3ec57dc7-81b7-474c-803d-60ec7988cec6',
 12: '60dde908-80a6-455d-85ff-3f439a331aac',
 13: '16fcc10e-c324-4f69-8834-baefc73e68c3',
 14: 'e0e0e3e2-c7e0-49fa-9f44-a2be33cf6504',
 15: '1031aa92-1590-413c-bbea-2d11e3739029',
 16: '89cbfdf8-f13d-4121-9a2b-e0f507acd944',
 17: 'ec1e23f2-e130-4fa7-ba31-3cbf495a3c94',
 18: '0b6f7de5-8fad-4bc5-88bf-954af287b7b3',
 19: '3935ff73-bd55-4110-87ee-24c6aa589b57',
 20: '07f98db0-7fd5-498d-b0c7-5aa916d8363b',
 21: '31d544fe-650f-4f29-95bc-592964a76ba7',
 22: '0d0d1c5c-b3de-

In [36]:
vector_store.get_by_ids(['07f98db0-7fd5-498d-b0c7-5aa916d8363b'])

[Document(id='07f98db0-7fd5-498d-b0c7-5aa916d8363b', metadata={}, page_content="yeah it made you made you realize made me realize that you can sort of in the way in the choices you make can define uh the where you end up and that means all of us are capable of the good uh evil it all matters in uh the different choices along the trajectory to those places that you make it's fascinating i mean games can do that philosophically to you and it's rare it seems rare yeah well games are i think a unique medium because um you as the player you're not just passively consuming the the entertainment right you're actually actively involved as an as a as an agent so i think that's what makes it in some ways can be more visceral than other other mediums like you know films and books so the second so that was you know designing ai and games and then the third use uh uh i've we've used of ai is in deep mind from the beginning which is using games as a testing ground for proving out ai algorithms and d

## Step 2 - Retrieval

In [15]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [16]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEndpointEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x784ee4104a10>, search_kwargs={'k': 4})

In [17]:
retriever.invoke('What is deepmind')

[Document(id='257a1faa-7fd2-45a5-a37c-3a3a49f6dedf', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

## Step 3 - Augmentation

In [18]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

In [19]:
# DOWNLOAD THE MODEL LOCALLY USING pipeline+HuggingFacePipeline
# generator = pipeline(
#     "text2text-generation",
#     model="google/flan-t5-base",
#     device=-1,
#     max_new_tokens=512
# )
# llm = HuggingFacePipeline(pipeline=generator)

from langchain_huggingface import HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="conversational"   # instead of "text-generation"
)
model = ChatHuggingFace(llm=llm)

In [20]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [21]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [22]:
retrieved_docs

[Document(id='45201fc9-5f72-4815-8ff8-3fe72e3b07a4', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [23]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of\n\

In [24]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [25]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we 

## Step 4 - Generation

In [26]:
answer = model.invoke(final_prompt)
print(answer.content)

Yes, the topic of nuclear fusion is discussed. 

They discussed how AI can improve the understanding and manipulation of various aspects of fusion energy:

* Aiming to identify "bottleneck problems" hindering progress in fusion.
* Collaborating with EPFL, a Swiss technical institute, to utilize their test reactor.
* Working on controlling plasma in the form of "droplets."
* Using Deep Reinforcement Learning (DRL) for control of high temperature plasmas. 


There is a significant discussion about how AI could be used to accelerate research related to nuclear fusion. 



## Building a Chain

In [27]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [28]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [29]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [30]:
parallel_chain.invoke('who is Demis')

{'context': "to get world peace because there's also other corrupting things like wanting power over people and this kind of stuff which is not necessarily satisfied by by just abundance but i think it will help um and i think uh but i think ultimately ai is not going to be run by any one person or one organization i think it should belong to the world belong to humanity um and i think maybe many there'll be many ways this will happen and ultimately um everybody should have a say in that do you have advice for uh young people in high school and college maybe um if they're interested in ai or interested in having a big impact on the world what they should do to have a career they can be proud of her to have a life they can be proud of i love giving talks to the next generation what i say to them is actually two things i i think the most important things to learn about and to find out about when you're when you're young is what are your true passions is first of all there's two things on

In [31]:
parser = StrOutputParser()

In [34]:
main_chain = parallel_chain | prompt | model | parser

In [35]:
main_chain.invoke('Can you summarize the video')

'The speaker believes a deeper, simpler explanation for the universe is needed beyond the current Standard Model.  They argue it would address mysteries like consciousness, life, and gravity.  The speaker use examples like Paul Dirac and Richard Feynman, pioneers who simplified complex concepts to better understand physics and the universe. The speaker also suggests that AI, in trying to explain things to humans, actually demonstrates that humans are better at explaining things simply because they might be less intelligent by nature.  While highlighting the importance of human interaction, the speaker emphasizes that ultimately, we must find new ways to express our understanding of the universe in a way that is accessible to everyone. \n'